# Gemma3:4b Image Phrase Extractor — Google Colab (Google Drive)

**Images are read directly from Google Drive.** No ZIP upload needed.

### Before running:

1. Go to [drive.google.com](https://drive.google.com) and upload your image folder into **My Drive**
2. Set `DRIVE_IMAGE_FOLDER` in Cell 1 to the exact folder name you uploaded
3. **Allow popups** for `colab.research.google.com` in your browser — the Drive mount opens a Google sign-in popup

**If Cell 1 fails with a credential error:**
- Allow popups in your browser settings and re-run Cell 1
- Or try: `Runtime → Restart session` then re-run

**Runtime recommendation:** `Runtime → Change runtime type → T4 GPU` (free tier) for faster inference.

In [ ]:
# ── Cell 1: Mount Google Drive and set image folder ───────────────────────────
from google.colab import auth, drive
import os, zipfile

# Explicitly authenticate with your Google account first (opens a popup)
auth.authenticate_user()

# Then mount Drive
drive.mount('/content/drive', force_remount=True)

# ↓ Change this to the name of the folder you uploaded to My Drive
DRIVE_IMAGE_FOLDER = 'vqaimages'

DRIVE_DIR = f'/content/drive/MyDrive/{DRIVE_IMAGE_FOLDER}'

assert os.path.isdir(DRIVE_DIR), (
    f'Folder not found: {DRIVE_DIR}\n'
    f'Make sure "{DRIVE_IMAGE_FOLDER}" exists inside My Drive on Google Drive.'
)

# If the folder contains ZIP files, extract them locally (Drive ZIPs can't be scanned directly)
zip_files = [f for f in os.listdir(DRIVE_DIR) if f.lower().endswith('.zip')]
if zip_files:
    EXTRACT_DIR = f'/content/{DRIVE_IMAGE_FOLDER}_extracted'
    os.makedirs(EXTRACT_DIR, exist_ok=True)
    for zf in zip_files:
        print(f'Extracting {zf} ...')
        with zipfile.ZipFile(os.path.join(DRIVE_DIR, zf), 'r') as z:
            z.extractall(EXTRACT_DIR)
    IMAGE_DIR = EXTRACT_DIR
    print(f'Extracted to {IMAGE_DIR}')
else:
    IMAGE_DIR = DRIVE_DIR
    print(f'Reading images directly from {IMAGE_DIR}')

total = sum(1 for _, _, fs in os.walk(IMAGE_DIR) for f in fs)
print(f'Found {total} file(s) in total')

In [ ]:
# ── Cell 2: Install Ollama + Python dependencies ──────────────────────────────
!apt-get install -qq zstd       # required by Ollama installer
!curl -fsSL https://ollama.com/install.sh | sh
!pip install -q requests Pillow opencv-python-headless numpy

In [ ]:
# ── Cell 3: Start Ollama server in background ─────────────────────────────────
import subprocess, time, requests

proc = subprocess.Popen(
    ['ollama', 'serve'],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)

for _ in range(20):
    try:
        requests.get('http://127.0.0.1:11434/api/tags', timeout=2)
        print('Ollama server is running.')
        break
    except Exception:
        time.sleep(1)
else:
    raise RuntimeError('Ollama server did not start in time.')

In [ ]:
# ── Cell 4: Pull gemma3:4b ────────────────────────────────────────────────────
# Downloads ~3.3 GB — takes a few minutes.
!ollama pull gemma3:4b

In [ ]:
# ── Cell 5: Configuration ─────────────────────────────────────────────────────
OLLAMA_URL       = 'http://127.0.0.1:11434/api/generate'
MODEL_NAME       = 'gemma3:4b'
OLLAMA_TIMEOUT   = 600
MAX_IMAGE_PX     = 512
FRAMES_PER_VIDEO = 4

IMAGE_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp', '.tif', '.tiff'}
VIDEO_EXTS = {'.mp4', '.avi', '.mov', '.mkv', '.wmv', '.m4v'}

PROMPT = (
    'Look at this image carefully. '
    'Describe it in ONE short phrase (5-10 words) focusing on: '
    'whether fire is present, its nature (e.g. wildfire, campfire, industrial flame, explosion), '
    'and the surrounding environment. '
    'Reply with the phrase only — no extra text.'
)

In [ ]:
# ── Cell 6: Helper functions ──────────────────────────────────────────────────
import base64, json, time
from io import BytesIO
from pathlib import Path

import cv2
import numpy as np
import requests
from PIL import Image


def resize_image(img, max_px=MAX_IMAGE_PX):
    w, h = img.size
    if max(w, h) <= max_px:
        return img
    scale = max_px / max(w, h)
    return img.resize((int(w * scale), int(h * scale)), Image.LANCZOS)


def image_to_b64(path):
    img = resize_image(Image.open(path).convert('RGB'))
    buf = BytesIO()
    img.save(buf, format='JPEG', quality=85)
    return base64.b64encode(buf.getvalue()).decode('utf-8')


def pil_to_b64(img):
    img = resize_image(img.convert('RGB'))
    buf = BytesIO()
    img.save(buf, format='JPEG', quality=85)
    return base64.b64encode(buf.getvalue()).decode('utf-8')


def ollama_phrase(image_b64):
    payload = {
        'model':   MODEL_NAME,
        'prompt':  PROMPT,
        'images':  [image_b64],
        'stream':  True,
        'options': {'temperature': 0.2},
    }
    tokens = []
    with requests.post(OLLAMA_URL, json=payload, stream=True, timeout=OLLAMA_TIMEOUT) as resp:
        resp.raise_for_status()
        for line in resp.iter_lines():
            if not line:
                continue
            data = json.loads(line.decode('utf-8') if isinstance(line, bytes) else line)
            tokens.append(data.get('response', ''))
            if data.get('done'):
                break
    return ''.join(tokens).strip()


def extract_frames(video_path, n=FRAMES_PER_VIDEO):
    cap   = cv2.VideoCapture(str(video_path))
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total == 0:
        cap.release()
        return []
    frames = []
    for idx in np.linspace(0, total - 1, n, dtype=int):
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(idx))
        ok, frame = cap.read()
        if ok:
            frames.append(Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)))
    cap.release()
    return frames


print('Helper functions ready.')

In [ ]:
# ── Cell 7: Process all images / videos ───────────────────────────────────────
root     = Path(IMAGE_DIR)
out_path = Path('/content/drive/MyDrive/phrases.txt')  # saved back to Drive

all_files = sorted(
    p for p in root.rglob('*')
    if p.is_file() and p.suffix.lower() in IMAGE_EXTS | VIDEO_EXTS
)

print(f'Found {len(all_files)} file(s)\n' + '─' * 60)

with open(out_path, 'w') as out:
    for idx, path in enumerate(all_files, 1):
        rel = path.relative_to(root)
        ext = path.suffix.lower()
        print(f'[{idx}/{len(all_files)}] {rel}')

        try:
            t0 = time.time()

            if ext in IMAGE_EXTS:
                phrase = ollama_phrase(image_to_b64(path))
                print(f'  → {phrase!r}  ({time.time()-t0:.1f}s)')
                out.write(f'{rel} | {phrase}\n')

            else:
                frames = extract_frames(path)
                for i, frame in enumerate(frames):
                    phrase = ollama_phrase(pil_to_b64(frame))
                    print(f'  frame {i+1}/{len(frames)} → {phrase!r}  ({time.time()-t0:.1f}s)')
                    out.write(f'{rel} [frame {i+1}] | {phrase}\n')
                    t0 = time.time()

        except Exception as e:
            print(f'  ERROR: {e}')
            out.write(f'{rel} | ERROR: {e}\n')

        out.flush()

print(f'\nDone. phrases.txt saved to your Google Drive → My Drive/phrases.txt')